# GPT-2 residual stream — every block, streamed to HDF5

A decoder-only model, capturing all 12 blocks at once: 12 x seq_len x 768 floats
per passage. This is where keeping activations in RAM stops being reasonable and
`path=` starts to matter.

Downloads on first run: WikiText-2 (~5 MB) and GPT-2 weights (~500 MB).

In [1]:
from pathlib import Path

from transformers import AutoModel, AutoTokenizer

from nnact import ActivationMapper, H5ActivationStore
from nnact.utils import WikiTextSamples, activation_loader

CACHE = Path("gpt2_residual.h5")

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 ships without a pad token
model = AutoModel.from_pretrained("gpt2")  # not ...LMHeadModel: no logits needed


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [2]:
dataset = WikiTextSamples(tokenizer, n=512, max_length=64)
mapper = ActivationMapper(model)

# The residual stream: every block, plus the final layer norm.
LAYERS = [f"h.{i}" for i in range(12)] + ["ln_f"]
print(f"{len(dataset)} passages | {len(LAYERS)} layers: {LAYERS}")


512 passages | 13 layers: ['h.0', 'h.1', 'h.2', 'h.3', 'h.4', 'h.5', 'h.6', 'h.7', 'h.8', 'h.9', 'h.10', 'h.11', 'ln_f']


In [3]:
# Cost before committing: one sample gives the per-sample shape.
probe_loader = activation_loader(dataset, batch_size=1)
probe = mapper.map(probe_loader, LAYERS, progress=False).summary()
per_sample = probe["elements"].sum() * 4
print(f"{per_sample / 1024:.0f} KB per passage")
print(f"{per_sample * len(dataset) / 1024**2:.0f} MB for {len(dataset)} passages")
print(f"{per_sample * 100_000 / 1024**3:.1f} GB for 100k passages")


2496 KB per passage
1248 MB for 512 passages
238.0 GB for 100k passages


In [4]:
# path= streams to HDF5 instead of RAM: memory stays flat in the sample count,
# and the file outlives the session.
loader = activation_loader(dataset, batch_size=16)
store = mapper.map(loader, LAYERS, CACHE)
print(f"file on disk: {CACHE.stat().st_size / 1024**2:.1f} MB")
store.metadata


GPT2Model activations[1/32]   3%|3          [00:00<?]

file on disk: 1248.8 MB


RunMetadata(
    model      = GPT2Model
    parameters = 124.4M
    layers     = h.0, h.1, h.2, h.3, h.4, h.5, h.6, h.7, h.8, h.9, h.10, h.11, ln_f
    samples    = 512
    batch_size = 16
    device     = cpu
    seconds    = 8.16s
    throughput = 62/s
    created    = 2026-09-14T19:21:41+00:00
)

In [5]:
store.summary()

,shape,elements,bytes
layer,,,
h.0,"(64, 768)",49152,100663296
h.1,"(64, 768)",49152,100663296
h.2,"(64, 768)",49152,100663296
h.3,"(64, 768)",49152,100663296
h.4,"(64, 768)",49152,100663296
h.5,"(64, 768)",49152,100663296
h.6,"(64, 768)",49152,100663296
h.7,"(64, 768)",49152,100663296
h.8,"(64, 768)",49152,100663296


In [6]:
# Reads come off disk one sample at a time; nothing is held in RAM.
sample = store[0]
print(
    store.sample_ids[0],
    "->",
    len(sample.activations),
    "layers,",
    tuple(sample.activations[0].tensor.shape),
    "each\n",
)

# Residual stream norm grows with depth - the usual GPT-2 picture.
mask = dataset.encoded["attention_mask"][0].bool()
for act in sample.activations:
    real = act.tensor[mask]  # skip padding positions
    print(f"  {act.layer_name:<6} mean L2 norm = {real.norm(dim=-1).mean():6.2f}")

wiki_0000 -> 13 layers, (64, 768) each

  h.0    mean L2 norm =  55.14
  h.1    mean L2 norm =  66.87
  h.2    mean L2 norm = 101.30
  h.3    mean L2 norm = 109.17
  h.4    mean L2 norm = 116.07
  h.5    mean L2 norm = 123.46
  h.6    mean L2 norm = 132.14
  h.7    mean L2 norm = 146.98
  h.8    mean L2 norm = 165.41
  h.9    mean L2 norm = 195.88
  h.10   mean L2 norm = 269.62
  h.11   mean L2 norm = 453.89
  ln_f   mean L2 norm = 217.58


In [7]:
store.close()

# Reopening verifies the cache matches the dataset it was built from: ids are
# hashed in order, so a reordered or different dataset is rejected here rather
# than silently misaligning.
reloaded = H5ActivationStore.load(CACHE, [f"wiki_{i:04d}" for i in range(len(dataset))])
print(f"reloaded {len(reloaded)} samples, {len(reloaded.layer_names)} layers")
print(reloaded.metadata)  # travels with the file
reloaded.close()

CACHE.unlink(missing_ok=True)  # drop this line to keep the cache

reloaded 512 samples, 13 layers
RunMetadata(
    model      = GPT2Model
    parameters = 124.4M
    layers     = h.0, h.1, h.2, h.3, h.4, h.5, h.6, h.7, h.8, h.9, h.10, h.11, ln_f
    samples    = 512
    batch_size = 16
    device     = cpu
    seconds    = 8.16s
    throughput = 62/s
    created    = 2026-09-14T19:21:41+00:00
)
